# BioListen VN — Zenodo (InsectSet459) Exploratory Data Analysis (EDA) [Direct Zip-Reading]

Notebook này thực hiện phân tích khám phá dữ liệu (EDA) cho bộ dữ liệu **Zenodo InsectSet459** phục vụ việc mở rộng tập dữ liệu cho nhánh phân loại loài sinh vật (`species_head`).

### 💡 Giải pháp tối ưu hóa đĩa cứng (Direct In-Memory Zip-Reading):
Do bộ dữ liệu côn trùng InsectSet459 có dung lượng khổng lồ (~26,000 tệp tin WAV chia làm 2 file zip lớn), việc giải nén toàn bộ chúng ra đĩa cứng của Google Colab sẽ lập tức gây tràn không gian đĩa. Notebook này sử dụng phương pháp **giải nén nhị phân trực tiếp trên bộ nhớ RAM (In-memory Zip-Reading)**:
- Nạp tệp metadata CSV (`InsectSet459_Train_Val_Test_Annotation.csv`) trực tiếp từ Google Drive vào RAM.
- Lập chỉ mục song song tất cả các tệp tin có trong cả 2 tệp ZIP (`insect_Train.zip` và `insect_Validation.zip`) mà hoàn toàn không copy hay giải nén tệp zip ra ổ cứng Colab local.
- Khi thực hiện thống kê hoặc trực quan hóa mẫu âm thanh, hệ thống sẽ mở tệp ZIP từ Drive, đọc dữ liệu của file WAV đó **dưới dạng BytesIO trực tiếp trên RAM**, sau đó nạp qua các hàm xử lý âm thanh mà không ghi bất kỳ dữ liệu nào ra đĩa cứng của Colab.

## 1. Kết nối Google Drive & Khai báo Đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import io

# Khai báo các đường dẫn trực tiếp trên Google Drive
drive_raw_dir = '/content/drive/MyDrive/Datasets/BioListenVN/raw_zips'
zip_train_path = os.path.join(drive_raw_dir, 'insect_Train.zip')
zip_val_path = os.path.join(drive_raw_dir, 'insect_Validation.zip')
csv_path = os.path.join(drive_raw_dir, 'InsectSet459_Train_Val_Test_Annotation.csv')

print("Tập tin CSV tồn tại:", os.path.exists(csv_path))
print("Zip Train tồn tại:", os.path.exists(zip_train_path))
print("Zip Validation tồn tại:", os.path.exists(zip_val_path))

## 2. Đọc Metadata CSV trực tiếp từ Drive

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Nạp thành công metadata. Kích thước DataFrame: {df.shape}")
    print("\n5 dòng đầu tiên:")
    display(df.head())
    print("\nThông tin các cột:")
    df.info()
else:
    print("LỖI: Không tìm thấy file CSV metadata!")

## 3. Lập chỉ mục các file trong cả hai tệp ZIP

In [ ]:
file_to_zip = {}  # Ánh xạ tên file WAV thành bộ (zip_path, member_path)

def index_zip(zip_p):
    if os.path.exists(zip_p):
        print(f"Đang lập chỉ mục {os.path.basename(zip_p)}...")
        with zipfile.ZipFile(zip_p, 'r') as z:
            for p in z.namelist():
                if p.endswith('.wav'):
                    file_to_zip[os.path.basename(p)] = (zip_p, p)

index_zip(zip_train_path)
index_zip(zip_val_path)
print(f"Đã chỉ mục tổng cộng {len(file_to_zip)} file WAV từ cả 2 zip.")

# Tìm cột nhãn phù hợp
label_col = None
for col in ['species', 'label', 'class_name', 'Class Name', 'species_id']:
    if col in df.columns:
        label_col = col
        break

if not label_col and len(df.columns) > 0:
    label_col = df.columns[-1]

if label_col:
    class_counts = df[label_col].value_counts()
    print(f"Phân phối số lượng mẫu của các loài (theo cột '{label_col}'):")
    print(class_counts.head(20))
    
    plt.figure(figsize=(14, 8))
    sns.barplot(x=class_counts.values[:30], y=class_counts.index[:30], hue=class_counts.index[:30], palette="magma", legend=False)
    plt.title("Phân phối số lượng mẫu của Top 30 Loài Côn trùng (InsectSet459)", fontsize=14, fontweight='bold')
    plt.xlabel("Số lượng mẫu")
    plt.ylabel("Nhãn loài")
    plt.show()

## 4. Thống kê đặc tính vật lý âm học trong bộ nhớ

In [ ]:
import tqdm
import soundfile as sf
import librosa

# Lấy mẫu ngẫu nhiên 50 tệp để kiểm tra nhanh
sample_size = min(50, len(df))
sample_df = df.sample(sample_size, random_state=42)

durations = []
sample_rates = []
channels = []

# Sử dụng cache ZIP handles tránh mở/đóng liên tục
z_handles = {}

try:
    for idx, row in tqdm.tqdm(sample_df.iterrows(), total=len(sample_df)):
        fname = row['file_name'] if 'file_name' in row else (row['filename'] if 'filename' in row else row.iloc[0])
        
        if fname in file_to_zip:
            zip_p, member_p = file_to_zip[fname]
            if zip_p not in z_handles:
                z_handles[zip_p] = zipfile.ZipFile(zip_p, 'r')
                
            z = z_handles[zip_p]
            try:
                with z.open(member_p) as f:
                    # Đọc trực tiếp vào BytesIO RAM
                    data_bytes = io.BytesIO(f.read())
                    info = sf.info(data_bytes)
                    sample_rates.append(info.samplerate)
                    durations.append(info.duration)
                    channels.append(info.channels)
            except Exception as e:
                print(f"Lỗi đọc file {fname} từ ZIP: {e}")
finally:
    # Đóng toàn bộ ZIP handles giải phóng RAM
    for zh in z_handles.values():
        zh.close()

if len(durations) > 0:
    print("\n--- KẾT QUẢ PHÂN TÍCH NHANH 50 FILE INSECTSET459 (IN-MEMORY) ---")
    print(f"Tần số lấy mẫu (Sample Rates): {set(sample_rates)} Hz")
    print(f"Số kênh (Channels): {set(channels)}")
    print(f"Thời lượng ngắn nhất: {min(durations):.2f} giây")
    print(f"Thời lượng dài nhất: {max(durations):.2f} giây")
    print(f"Thời lượng trung bình: {np.mean(durations):.2f} giây")
    
    plt.figure(figsize=(8, 4))
    sns.histplot(durations, bins=15, kde=True, color='purple')
    plt.title("Phân phối thời lượng tệp tin (InsectSet459)")
    plt.xlabel("Thời lượng (giây)")
    plt.ylabel("Tần suất")
    plt.show()
else:
    print("Không đọc được thông tin âm thanh nào từ các tệp thử nghiệm.")

## 5. Trực quan hóa Âm thanh côn trùng tần số cao

In [ ]:
import librosa.display
import IPython.display as ipd

def plot_insect_call_from_zip(zip_path, zip_member, label):
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(zip_member) as f:
            y, sr = sf.read(io.BytesIO(f.read()))
            if len(y.shape) > 1:
                y = y[:, 0]
                
    duration = len(y) / sr
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    
    fig, ax = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f"Tiếng kêu côn trùng: {label} | File: {os.path.basename(zip_member)}", fontsize=14, fontweight='bold')
    
    librosa.display.waveshow(y, sr=sr, ax=ax[0], color='m')
    ax[0].set_title("Dạng sóng (Waveform)")
    ax[0].set_ylabel("Biên độ")
    
    img = librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis='time', y_axis='mel', ax=ax[1], cmap='magma')
    ax[1].set_title("Log-Mel Spectrogram (dB Scale)")
    fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
    
    plt.tight_layout()
    plt.show()
    
    display(ipd.Audio(y, rate=sr))

# Chọn một dòng ngẫu nhiên để vẽ
test_row = df.sample(1).iloc[0]
fname = test_row['file_name'] if 'file_name' in test_row else (test_row['filename'] if 'filename' in test_row else test_row.iloc[0])

if fname in file_to_zip:
    zip_path_target, member_path_target = file_to_zip[fname]
    plot_insect_call_from_zip(zip_path_target, member_path_target, test_row[label_col])
else:
    print(f"Không tìm thấy đường dẫn tệp trong ZIP cho: {fname}")